<a href="https://colab.research.google.com/github/Anto-sujin/spam_mail_detection/blob/main/spam_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SMS Spam Detection Using Machine Learning and NLP

This project focuses on building a robust machine learning model to classify SMS messages as either legitimate (ham) or unwanted (spam). Leveraging natural language processing (NLP) techniques, the goal is to develop an effective spam detection system that can help users filter out unsolicited messages, thereby improving communication efficiency and security. The project involves data preprocessing, feature engineering using TF-IDF, training and evaluating multiple classification algorithms, and hyperparameter tuning to select the best-performing model.

## 2. Problem Statement

Spam messages are unsolicited and often deceptive messages sent to a large number of recipients, typically for commercial purposes, phishing, or spreading malware. These messages can be annoying, time-consuming, and potentially harmful to users.

Automatic spam detection is crucial for several reasons:
- **User Experience:** Reduces clutter in inboxes, making it easier for users to find important messages.
- **Security:** Protects users from phishing attempts, scams, and malicious content.
- **Efficiency:** Saves users time by automatically filtering irrelevant messages.

This problem is framed as a **binary classification problem** because each SMS message must be categorized into one of two distinct classes:
- **0 = Ham:** Represents a legitimate or non-spam message.
- **1 = Spam:** Represents an unsolicited or unwanted message.

## 3. Project Objectives

The main objectives of this project are to:

- **Clean and Preprocess SMS Text:** Prepare the raw SMS message data for machine learning by performing tasks such as lowercasing, removing punctuation, and handling other text inconsistencies.
- **Convert Text into Numerical Features using TF-IDF:** Transform the textual content of SMS messages into a numerical representation that machine learning models can understand, specifically using the Term Frequency-Inverse Document Frequency (TF-IDF) method.
- **Train Multiple Classification Algorithms:** Implement and train various machine learning models suitable for text classification, such as Logistic Regression, Naive Bayes, Support Vector Machines (SVM), Decision Trees, Random Forests, Gradient Boosting, XGBoost, and LightGBM.
- **Compare Model Performance:** Evaluate the initial performance of all trained models using a standard set of metrics to identify promising candidates.
- **Tune the Best-Performing Models:** Optimize the hyperparameters of the top-performing models using techniques like GridSearchCV to enhance their predictive accuracy and generalization capabilities.
- **Select a Final Model:** Choose the best model based on comprehensive evaluation, considering both performance metrics and practical considerations.
- **Test the Final Model on Unseen Messages:** Validate the final model's effectiveness on new, previously unseen SMS messages to ensure its robustness and real-world applicability.

## 4. Dataset

This project utilizes the `SMSSpamCollection` dataset, which is a collection of SMS messages tagged as either 'ham' (legitimate) or 'spam'.

- **Dataset Name/Source:** `SMSSpamCollection`
- **Number of Samples:** 5572 messages (as shown by `df.shape` output `(5572, 2)`)
- **Features/Columns:**
    - `label`: The target variable, indicating whether a message is 'ham' or 'spam'.
    - `message`: The actual text content of the SMS message.
- **Target Variable:** `label`
- **Class Labels (Original):** 'ham', 'spam'
- **Class Labels (Encoded):**
    - `0` for 'ham'
    - `1` for 'spam'
- **Class Distribution:**
    - **Original:** `ham`: 4825 messages, `spam`: 747 messages (as shown by `df['label'].value_counts()`)
    - **Encoded:** `0`: 4825 messages, `1`: 747 messages (as shown by `df['label'].value_counts()` after mapping)

**Relevant Observations:**
- The dataset contains 5572 entries with no missing values in either the 'label' or 'message' columns.
- The dataset exhibits a significant class imbalance, with 'ham' messages being much more frequent than 'spam' messages. This imbalance is important to consider during model evaluation, favoring metrics like precision, recall, and F1-score over simple accuracy.

In [63]:
import pandas as pd

In [64]:
df=pd.read_csv("/content/SMSSpamCollection",sep="\t",header=None,names=["label", "message"])

In [65]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [66]:
df.shape

(5572, 2)

In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


This output from `df.info()` provides a summary of the DataFrame, including the column names, number of non-null values, and data types. It confirms that there are 5572 entries and two columns, 'label' and 'message', both initially of object (string) type, and no missing values, which is crucial for data quality.

The output `df['label'].value_counts()` shows the distribution of 'ham' and 'spam' messages in the dataset. It reveals a class imbalance, with 4825 'ham' messages and 747 'spam' messages. This imbalance indicates that the majority class ('ham') is significantly more represented than the minority class ('spam'). This is an important consideration when evaluating model performance, as a model could achieve high accuracy by simply predicting 'ham' for most messages if not properly handled or evaluated with appropriate metrics.

In [68]:
df['label'].value_counts()

,count
label,
ham,4825
spam,747


In [69]:
mapping ={"ham":0,"spam":1}

In [70]:
df['label']=df['label'].map(mapping)

In [71]:
df['label'].value_counts()

,count
label,
0,4825
1,747


In [72]:
df.isnull().sum()

,0
label,0
message,0


In [73]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   int64 
 1   message  5572 non-null   object
dtypes: int64(1), object(1)
memory usage: 87.2+ KB


This output from `df.info()` after label mapping and checking for null values shows that the 'label' column has been successfully converted from an `object` type to an `int64` type, reflecting the numerical encoding of 'ham' (0) and 'spam' (1). The 'message' column remains an `object` type, as expected for text data. Critically, the `Non-Null Count` remains 5572 for both columns, confirming that no missing values were introduced or are present, which is essential for model training.

In [74]:
df['message']=df['message'].str.lower().str.strip()

In [75]:
df.head()

,label,message
0,0,"go until jurong point, crazy.. available only ..."
1,0,ok lar... joking wif u oni...
2,1,free entry in 2 a wkly comp to win fa cup fina...
3,0,u dun say so early hor... u c already then say...
4,0,"nah i don't think he goes to usf, he lives aro..."


This `df.head()` output displays the first five rows of the DataFrame after the message column has been transformed. All messages are now in lowercase, and leading/trailing whitespace has been removed. This step ensures text consistency, reducing the feature space for the model and treating, for example, 'Free' and 'free' as the same word, which is important for accurate text analysis.

In [76]:
import string

In [77]:
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [78]:
df['message']=df['message'].replace(r"[^\w\s]","",regex=True)

## 6. Data Preprocessing

Data preprocessing is a critical step in preparing raw text data for machine learning models. It involves cleaning and transforming the text to a format that algorithms can understand and process effectively. The following steps were performed:

1.  **Label Encoding:**
    -   `mapping ={"ham":0,"spam":1}`: A dictionary was created to map the categorical labels 'ham' and 'spam' to numerical values 0 and 1, respectively.
    -   `df['label']=df['label'].map(mapping)`: This line applies the mapping to the 'label' column, converting text labels into numerical representations.
    -   **Why:** Machine learning models require numerical input. Encoding categorical labels into numerical format allows the models to process the target variable. This also establishes a clear numerical representation for binary classification.

2.  **Text Normalization (Lowercasing and Stripping Whitespace):**
    -   `df['message']=df['message'].str.lower().str.strip()`: All characters in the 'message' column were converted to lowercase, and any leading or trailing whitespace was removed.
    -   **Why:** Lowercasing ensures that words like 'Free' and 'free' are treated as the same, reducing vocabulary size and improving consistency. Stripping whitespace removes unnecessary characters that could be misinterpreted as part of the text, standardizing the input.

3.  **Punctuation Removal:**
    -   `df['message']=df['message'].replace(r"[^a-z0-9 ]","",regex=True)`: This line removes all punctuation marks from the 'message' column, retaining only alphanumeric characters and spaces.
    -   **Why:** Punctuation usually does not carry significant semantic meaning in spam detection and can add noise to the feature space. Removing it helps in focusing on the core words and reduces the complexity for the TF-IDF vectorizer.

In [79]:
from sklearn.model_selection import train_test_split

In [80]:
X=df.iloc[:,-1]
y=df.iloc[:,:-1]

In [81]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

## 7. Train-Test Split

The dataset is divided into training and testing sets to evaluate the model's performance on unseen data. This split helps in assessing how well the model generalizes to new examples and prevents overfitting.

-   **Why the dataset is split:** To simulate a real-world scenario where the model needs to make predictions on messages it has never encountered before. The training set is used to teach the model, and the test set is used to evaluate its predictive power.
-   **Training Set vs. Test Set:**
    -   **Training Set (`X_train`, `y_train`):** Used to train the machine learning model. The model learns patterns and relationships from this data.
    -   **Test Set (`X_test`, `y_test`):** Used to evaluate the trained model's performance. This data is kept separate and untouched during the training phase to provide an unbiased assessment of the model's generalization ability.
-   **Test Size:** `test_size=0.20` indicates that 20% of the data will be allocated to the test set, and the remaining 80% will be used for training.
-   **`random_state=42`:** This parameter ensures reproducibility. By setting a specific `random_state`, the data split will be the same every time the code is executed, which is crucial for consistent experimentation and debugging.
-   **`stratify` (implicitly handled by `y` being passed):** While not explicitly set to `stratify=y`, the `value_counts` checks confirm that the class distribution of `spam` and `ham` is preserved in both the training and testing sets, which is good practice for imbalanced datasets.

**Class Distribution in Splits:**

-   **Training Set (`y_train`):
    -   Total Samples: 4457
    -   Not Spam (0): 3859 (approximately 86%)
    -   Spam (1): 598 (approximately 13%)

-   **Test Set (`y_test`):
    -   Total Samples: 1115
    -   Not Spam (0): 966 (approximately 86%)
    -   Spam (1): 149 (approximately 13%)

These distributions show that the class imbalance present in the original dataset is maintained in both the training and test sets, which is important for realistic model evaluation.

In [82]:
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 4457
Testing samples: 1115


In [83]:
a = len(y_train)
b = y_train.value_counts()[0]
c = y_train.value_counts()[1]

In [84]:
print(f'NOT SPAM PERCENTAGE:{int((b/a)*100)}')

NOT SPAM PERCENTAGE:86


In [85]:
print(f'SPAM PERCENTAGE :{(int((c/a)*100))}')

SPAM PERCENTAGE :13


In [86]:
a = len(y_test)
b = y_test.value_counts()[0]
c = y_test.value_counts()[1]


In [87]:
print(int((b/a)*100))

86


In [88]:
print(int((c/a)*100))

13


These print statements confirm that the class distribution (ham vs. spam) is consistent between the training and testing sets, both showing approximately 86% ham and 13% spam. This indicates that the `train_test_split` successfully maintained the original class proportions, which is important for reliable model training and evaluation, especially with imbalanced datasets.

In [89]:
y_test.value_counts()

,count
label,
0,966
1,149


In [90]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer()
X_trainD=vec.fit_transform(X_train)
X_testD = vec.transform(X_test)

## 8. TF-IDF Feature Extraction

Text data cannot be directly fed into machine learning models. It needs to be converted into numerical representations. TF-IDF (Term Frequency-Inverse Document Frequency) is a widely used technique for this purpose, particularly effective in text classification tasks like spam detection.

**1. Term Frequency (TF):**
-   Measures how frequently a term (word) appears in a document (SMS message).
-   A higher TF value for a word indicates that the word is more important within that specific document.
-   Formula: $TF(t, d) = \text{Number of times term } t \text{ appears in document } d / \text{Total number of terms in document } d$

**2. Inverse Document Frequency (IDF):**
-   Measures the importance of a term across the entire corpus (all SMS messages).
-   Words that appear frequently in many documents (like

In [91]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


In [92]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

In [93]:
metrics = [
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
]

In [94]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Naive Bayes": MultinomialNB(),
    "Support Vector Machine": SVC(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(),
    "LightGBM": LGBMClassifier(),

}

In [95]:
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

In [96]:
 results = []

for model_name, model in models.items():

    model.fit(X_trainD, y_train)

    pre = model.predict(X_testD)

    if hasattr(model, "predict_proba"):
        score = model.predict_proba(X_testD)[:, 1]
    else:
        score = model.decision_function(X_testD)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, pre),
        "Precision": precision_score(y_test, pre),
        "Recall": recall_score(y_test, pre),
        "F1-Score": f1_score(y_test, pre),
        "ROC-AUC": roc_auc_score(y_test, score),
        "PR-AUC": average_precision_score(y_test, score)
    })

results_df = pd.DataFrame(results)



[LightGBM] [Info] Number of positive: 598, number of negative: 3859
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011909 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12795
[LightGBM] [Info] Number of data points in the train set: 4457, number of used features: 453
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.134171 -> initscore=-1.864573
[LightGBM] [Info] Start training from score -1.864573


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [97]:
print(results_df)

                    Model  Accuracy  Precision    Recall  F1-Score   ROC-AUC  \
0     Logistic Regression  0.968610   0.991379  0.771812  0.867925  0.987578   
1             Naive Bayes  0.956951   1.000000  0.677852  0.808000  0.974961   
2  Support Vector Machine  0.985650   1.000000  0.892617  0.943262  0.990913   
3           Decision Tree  0.955157   0.861314  0.791946  0.825175  0.886139   
4           Random Forest  0.976682   1.000000  0.825503  0.904412  0.995432   
5     K-Nearest Neighbors  0.921076   1.000000  0.409396  0.580952  0.961878   
6       Gradient Boosting  0.970404   0.983333  0.791946  0.877323  0.982422   
7                 XGBoost  0.974888   0.961832  0.845638  0.900000  0.978851   
8                LightGBM  0.982063   0.984962  0.879195  0.929078  0.982485   

     PR-AUC  
0  0.968730  
1  0.947342  
2  0.976664  
3  0.709917  
4  0.982890  
5  0.926376  
6  0.951273  
7  0.953442  
8  0.962698  


In [98]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV


In [99]:
# ==========================================
# 1. SVM Pipeline
# ==========================================

svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("model", SVC())
])

svm_param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 3],
    "tfidf__max_df": [0.95, 1.0],
    "tfidf__sublinear_tf": [False, True],

    "model__C": [0.1, 1, 10],
    "model__kernel": ["linear", "rbf"]
}

In [100]:
svm_grid = GridSearchCV(
    svm_pipeline,
    svm_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

In [101]:
# ==========================================
# 2. Random Forest Pipeline
# ==========================================

rf_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("model", RandomForestClassifier(random_state=42))
])

rf_param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 3],
    "tfidf__max_df": [0.95, 1.0],
    "tfidf__sublinear_tf": [False, True],

    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 20],
    "model__min_samples_split": [2, 5]
}
rf_grid = GridSearchCV(
    rf_pipeline,
    rf_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

In [102]:
# ==========================================
# 3. LightGBM Pipeline
# ==========================================

lgbm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("model", LGBMClassifier(
        random_state=42,
        verbosity=-1
    ))
])

lgbm_param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 3],
    "tfidf__max_df": [0.95, 1.0],
    "tfidf__sublinear_tf": [False, True],

    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.05, 0.1],
    "model__num_leaves": [31, 50]
}


# ==========================================
# 4. GridSearchCV
# ==========================================





lgbm_grid = GridSearchCV(
    lgbm_pipeline,
    lgbm_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)


In [103]:
svm_grid.fit(X_train, y_train)
rf_grid.fit(X_train, y_train)
lgbm_grid.fit(X_train, y_train)

Fitting 5 folds for each of 144 candidates, totalling 720 fits
Fitting 5 folds for each of 192 candidates, totalling 960 fits
Fitting 5 folds for each of 192 candidates, totalling 960 fits


/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tfidf', TfidfVectorizer()),
                                       ('model',
                                        LGBMClassifier(random_state=42,
                                                       verbosity=-1))]),
             n_jobs=-1,
             param_grid={'model__learning_rate': [0.05, 0.1],
                         'model__n_estimators': [100, 200],
                         'model__num_leaves': [31, 50],
                         'tfidf__max_df': [0.95, 1.0],
                         'tfidf__min_df': [1, 2, 3],
                         'tfidf__ngram_range': [(1, 1), (1, 2)],
                         'tfidf__sublinear_tf': [False, True]},
             scoring='f1', verbose=1)

In [104]:
print("\n===== SVM =====")
print("Best Parameters:", svm_grid.best_params_)
print("Best CV F1:", svm_grid.best_score_)

print("\n===== Random Forest =====")
print("Best Parameters:", rf_grid.best_params_)
print("Best CV F1:", rf_grid.best_score_)

print("\n===== LightGBM =====")
print("Best Parameters:", lgbm_grid.best_params_)
print("Best CV F1:", lgbm_grid.best_score_)


===== SVM =====
Best Parameters: {'model__C': 10, 'model__kernel': 'linear', 'tfidf__max_df': 0.95, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}
Best CV F1: 0.9444601894183874

===== Random Forest =====
Best Parameters: {'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 100, 'tfidf__max_df': 0.95, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 1), 'tfidf__sublinear_tf': False}
Best CV F1: 0.8909059146989051

===== LightGBM =====
Best Parameters: {'model__learning_rate': 0.1, 'model__n_estimators': 100, 'model__num_leaves': 31, 'tfidf__max_df': 0.95, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': False}
Best CV F1: 0.9099022489291893


In [105]:

print("\n===== Random Forest =====")
print("Best Parameters:", rf_grid.best_params_)
print("Best CV F1:", rf_grid.best_score_)



===== Random Forest =====
Best Parameters: {'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 100, 'tfidf__max_df': 0.95, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 1), 'tfidf__sublinear_tf': False}
Best CV F1: 0.8909059146989051


In [106]:

print("\n===== LightGBM =====")
print("Best Parameters:", lgbm_grid.best_params_)
print("Best CV F1:", lgbm_grid.best_score_)


===== LightGBM =====
Best Parameters: {'model__learning_rate': 0.1, 'model__n_estimators': 100, 'model__num_leaves': 31, 'tfidf__max_df': 0.95, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': False}
Best CV F1: 0.9099022489291893


In [ ]:
model = SVC(C=10,kernel='linear')
tfidf =TfidfVectorizer(max_df=0.95,min_df=2,ngram_range=(1,2),sublinear_tf=True)
X_trainF =tfidf.fit_transform(X_train)
X_testF =tfidf.transform(X_test)
model.fit(X_trainF,y_train)
pre=model.predict(X_testF)
score=model.decision_function(X_testF)


print(f"Accuracy: {accuracy_score(y_test, pre):.4f}")
print(f"Precision: {precision_score(y_test, pre):.4f}")
print(f"Recall: {recall_score(y_test, pre):.4f}")
print(f"F1-Score: {f1_score(y_test, pre):.4f}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, pre)}")
print(f"ROC-AUC: {roc_auc_score(y_test, score):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, score):.4f}")

In [ ]:

finial_model=svm_grid.best_estimator_


In [ ]:
finial_model

In [ ]:
finial_model.fit(X_train,y_train)

In [ ]:
import joblib

In [ ]:
joblib.dump(finial_model, "spam_detector.pkl")

print("Model saved successfully!")